In [ ]:
import json
import re
import time
import textwrap
from dataclasses import dataclass, field
from datetime import datetime
from urllib.parse import urlparse
from urllib import robotparser

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential
import ollama  

print("Ollama models currently available locally:")
for m in ollama.list().get("models", []):
    print(" -", m.get("model", m))

Ollama models currently available locally:
 - phi3:latest


In [ ]:


COMPETITOR_URLS = [
    "https://books.toscrape.com",
    "https://quotes.toscrape.com",
    
]

OLLAMA_MODEL = "phi3"             
                                  
REQUEST_TIMEOUT = 15              
USER_AGENT = "CompetitiveAnalysisBot/1.0 (+research use, respects robots.txt)"
MAX_CHARS_PER_PAGE = 12000        
OUTPUT_DIR = "reports"


In [ ]:


import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def is_scraping_allowed(url: str, user_agent: str = USER_AGENT) -> bool:
    """Check robots.txt before fetching a page."""
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = robotparser.RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        
        return True


@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_page(url: str) -> str:
    headers = {"User-Agent": USER_AGENT}
    resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.text


def extract_page_content(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "lxml")

    title = soup.title.string.strip() if soup.title and soup.title.string else ""

    meta_desc = ""
    meta_tag = soup.find("meta", attrs={"name": "description"})
    if meta_tag and meta_tag.get("content"):
        meta_desc = meta_tag["content"].strip()

    
    for tag in soup(["script", "style", "noscript", "svg", "img"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    
    price_hits = re.findall(r"[^.]{0,80}(?:\$\s?\d[\d,]*|pricing|/mo|/month|/year)[^.]{0,80}", text, flags=re.I)
    price_snippets = list(dict.fromkeys(price_hits))[:15]  

    return {
        "url": url,
        "title": title,
        "meta_description": meta_desc,
        "text": text[:MAX_CHARS_PER_PAGE],
        "price_snippets": price_snippets,
    }


def scrape_competitor(url: str) -> dict:
    if not is_scraping_allowed(url):
        print(f"⚠️  robots.txt disallows scraping {url} — skipping.")
        return {"url": url, "skipped": True}
    try:
        html = fetch_page(url)
        data = extract_page_content(html, url)
        data["skipped"] = False
        return data
    except Exception as e:
        print(f"❌ Failed to scrape {url}: {e}")
        return {"url": url, "skipped": True, "error": str(e)}


In [ ]:

assert "COMPETITOR_URLS" in dir(), "COMPETITOR_URLS is not defined — run Cell 3 (Configuration) first."

scraped_pages = []
for url in tqdm(COMPETITOR_URLS, desc="Scraping competitors"):
    scraped_pages.append(scrape_competitor(url))
    time.sleep(1) 


for p in scraped_pages:
    status = "SKIPPED" if p.get("skipped") else "OK"
    print(f"[{status}] {p['url']}")


Scraping competitors:   0%|          | 0/2 [00:00<?, ?it/s]

[OK] https://books.toscrape.com
[OK] https://quotes.toscrape.com


In [ ]:


EXTRACTION_SCHEMA = {
    "company_name": "string",
    "one_line_positioning": "string",
    "target_audience": "string",
    "pricing_model": "string (e.g. freemium, subscription tiers, usage-based, unknown)",
    "pricing_details": "string (any concrete numbers found, or 'not disclosed')",
    "key_features": ["string", "..."],
    "differentiators": ["string", "..."],
    "tone_and_voice": "string (e.g. playful, enterprise/formal, technical)",
    "calls_to_action": ["string", "..."],
    "notable_gaps_or_weaknesses": ["string", "..."]
}

EXTRACTION_SYSTEM_PROMPT = f"""You are a market research analyst. You will be given the scraped
text of a company's website. Extract the following fields and respond with ONLY valid JSON
matching this schema (no markdown fences, no commentary, no extra keys):

{json.dumps(EXTRACTION_SCHEMA, indent=2)}

If a field is not discoverable from the text, use "unknown" (or an empty list for list fields).
Never invent numbers or facts that aren't supported by the text."""


@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def extract_structured_data(page: dict, model: str = OLLAMA_MODEL) -> dict:
    if page.get("skipped"):
        return {"url": page["url"], "error": "skipped during scraping"}

    user_content = textwrap.dedent(f"""
        URL: {page['url']}
        Title: {page['title']}
        Meta description: {page['meta_description']}
        Pricing-looking snippets found on page: {page['price_snippets']}

        Page text:
        {page['text']}
    """)

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        format="json",  
        options={"temperature": 0.1},  
    )

    raw = response["message"]["content"]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
       
        match = re.search(r"\{.*\}", raw, flags=re.S)
        parsed = json.loads(match.group(0)) if match else {"error": "unparseable", "raw": raw}

    parsed["url"] = page["url"]
    return parsed


In [ ]:

structured_results = []
for page in tqdm(scraped_pages, desc="Extracting with local LLM"):
    structured_results.append(extract_structured_data(page))

df = pd.json_normalize(structured_results)
df


Extracting with local LLM:   0%|          | 0/2 [00:00<?, ?it/s]

,company_name,one_line_positioning,target_audience,pricing_model,pricing_details,key_features,differentiators,tone_and_voice,calls_to_action,notable_gaps_or_weaknesses,url
0,Books to Scrape - Sandbox,Demo website for web scraping purposes,Web scraping enthusiasts,unknown,Prices and ratings here were randomly assigned...,[Demo website for web scraping purposes],[Demo website for web scraping purposes],playful,[Add to basket],"[No real meaning prices and ratings, Limited t...",https://books.toscrape.com
1,unknown,unknown,unknown,unknown,not disclosed,"[Quotes to Scrape, Login]","[unknown, unknown]",unknown,"[unknown, unknown]","[unknown, unknown]",https://quotes.toscrape.com


In [ ]:


REPORT_SYSTEM_PROMPT = """You are a senior market research analyst producing a competitive
analysis report for an executive audience. You will be given structured JSON data extracted
from several competitor websites. Write a clear, well-organized Markdown report with these
sections:

1. Executive Summary (3-5 sentences)
2. Competitor Snapshot Table (markdown table: Company | Positioning | Pricing | Target Audience)
3. Feature Comparison (bullet list per competitor)
4. Market Positioning & Gaps (where competitors overlap, and where there's white space)
5. Strategic Recommendations (3-6 concrete, actionable bullets)

Be specific and reference the data given. Do not invent facts beyond what's provided. If data
for a competitor is thin, say so plainly rather than filling gaps with assumptions."""


def generate_market_report(results: list, model: str = OLLAMA_MODEL) -> str:
    data_block = json.dumps(results, indent=2)
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": REPORT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Competitor data:\n{data_block}"},
        ],
        options={"temperature": 0.4},
    )
    return response["message"]["content"]


report_markdown = generate_market_report(structured_results)
print(report_markdown)


# Executive Summary

This report provides a comprehensive analysis of two competitors in the web scraping and quote-gathering industry. Our findings reveal that both companies have undeveloped positioning, target audience, and pricing models, along with limited key features and differentiators. Notably, both exhibit gaps in their market strategies, particularly in terms of pricing clarity and service offerings.

# Competitor Snapshot Table

| Company Name                 | Positioning                                                                 | Pricing | Target Audience       |
|-----------------------------|----------------------------------------------------------------------------|---------|----------------------|
| Books to Scrape - Sandbox   | Demo website for web scraping purposes                                       | Unknown | Web scraping enthusiasts |
| Unknown                     | Unknown                                                                    | Unknown | U

In [ ]:


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
md_path = os.path.join(OUTPUT_DIR, f"competitive_analysis_{timestamp}.md")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(f"# Competitive Analysis Report\n")
    f.write(f"_Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} using local model `{OLLAMA_MODEL}`_\n\n")
    f.write(report_markdown)

print(f"Saved report to: {md_path}")


json_path = os.path.join(OUTPUT_DIR, f"competitor_data_{timestamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(structured_results, f, indent=2)
print(f"Saved raw structured data to: {json_path}")


Saved report to: reports\competitive_analysis_20260828_023404.md
Saved raw structured data to: reports\competitor_data_20260828_023404.json
